# Processing samples

In [1]:
# Cell 1: imports and base paths

import os
import numpy as np
import pandas as pd
from scipy.io import mmread
from scipy import sparse
import anndata as ad

# match your R base_dir
base_dir = os.path.expanduser("~/Roselab/Spatial/CAR_T/data/sc-reference/")

counts_file      = os.path.join(base_dir, "counts.mtx")
features_file    = os.path.join(base_dir, "features.tsv")
barcodes_file    = os.path.join(base_dir, "barcodes.tsv")
metadata_file    = os.path.join(base_dir, "metadata.csv")
cell_type_file   = os.path.join(base_dir, "cell_type.csv")
umap_file        = os.path.join(base_dir, "umap_coords.csv")

print(base_dir)

# Summary:
# - Set up file paths that match what you wrote out from Seurat.


/Users/janzules/Roselab/Spatial/CAR_T/data/sc-reference/


In [2]:
# Cell 2: load matrix + feature / barcode names

# counts.mtx is genes x cells (Seurat / 10x style) → transpose to cells x genes
X = mmread(counts_file).tocsr().T  # csr is convenient for AnnData
print("Raw matrix shape (cells x genes):", X.shape)

# gene names
features = pd.read_csv(
    features_file,
    header=None,
    sep="\t"
)
features.columns = ["gene_id"]
print("Features:", features.shape)

# barcodes (cell IDs)
barcodes = pd.read_csv(
    barcodes_file,
    header=None,
    sep="\t"
)
barcodes.columns = ["barcode"]
print("Barcodes:", barcodes.shape)

# sanity checks
assert X.shape[0] == barcodes.shape[0], "Cell dimension mismatch."
assert X.shape[1] == features.shape[0], "Gene dimension mismatch."

# Summary:
# - Loaded counts as a sparse matrix and transposed → cells x genes.
# - Loaded gene IDs and barcodes; dimensions are consistent.


Raw matrix shape (cells x genes): (135379, 22949)
Features: (22949, 1)
Barcodes: (135379, 1)


In [3]:
# Cell 3: load full metadata and explicit cell_type file, then merge

# metadata.csv from seurat_obj@meta.data
# write.csv in R saves rownames as a column named "row.names" by default
meta = pd.read_csv(metadata_file, index_col=0)
print("Metadata shape:", meta.shape)
print("Metadata index example:", meta.index[:5])

# Ensure metadata index matches barcodes
# If needed, reindex to the barcodes order
if not np.array_equal(meta.index.values, barcodes["barcode"].values):
    # try to align by barcode name
    meta = meta.reindex(barcodes["barcode"].values)
    print("Reindexed metadata to match barcodes.")
    assert np.all(meta.index.values == barcodes["barcode"].values), \
        "Metadata could not be aligned to barcodes."

# load explicit cell_type mapping (Barcode, cell_type)
cell_type_df = pd.read_csv(cell_type_file)
print("Cell type table shape:", cell_type_df.shape)

# set Barcode as index for easy join
cell_type_df = cell_type_df.set_index("Barcode")

# add cell_type as a column in metadata (name matches Seurat's sctype_classification)
if "cell_type" in cell_type_df.columns:
    meta["cell_type"] = cell_type_df.loc[meta.index, "cell_type"]
else:
    # if the column name is different for some reason, this makes it explicit
    meta["cell_type"] = cell_type_df.iloc[:, 0].reindex(meta.index)

# Summary:
# - Loaded full Seurat meta.data and aligned to barcodes.
# - Attached an explicit `cell_type` column (for cell2location).


Metadata shape: (135379, 14)
Metadata index example: Index(['56948_AAACGAACACTAGGTT-1', '56948_AAACGCTGTTGGGTTT-1',
       '56948_AAACGCTTCCTCGATC-1', '56948_AAAGAACCAATACAGA-1',
       '56948_AAAGAACCACAGGATG-1'],
      dtype='object')
Cell type table shape: (135379, 2)


In [4]:
# Cell 4: create AnnData with everything so far

adata = ad.AnnData(
    X=X,
    obs=meta.copy(),                # all cell-level metadata
    var=pd.DataFrame(index=features["gene_id"].values)  # genes
)

adata.obs_names = barcodes["barcode"].values
adata.var_names = features["gene_id"].values

print(adata)
print("obs columns:", list(adata.obs.columns)[:10])

# Summary:
# - Created AnnData with:
#   - X = raw counts (cells x genes, sparse)
#   - obs = full Seurat metadata + cell_type
#   - var = gene IDs


AnnData object with n_obs × n_vars = 135379 × 22949
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Barcode', 'Library', 'condition', 'file_name', 'percent.mt', 'unintegrated_clusters', 'seurat_clusters', 'harmony_clusters', 'sctype_classification', 'seurat.cluster.ann', 'treatment', 'cell_type'
obs columns: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Barcode', 'Library', 'condition', 'file_name', 'percent.mt', 'unintegrated_clusters', 'seurat_clusters']


In [5]:
# Cell 5: load UMAP from R and store as UMAP_R

umap_df = pd.read_csv(umap_file)
print("UMAP table shape:", umap_df.shape)
print(umap_df.head())

# Check that barcode column matches obs_names
assert "Barcode" in umap_df.columns, "UMAP file missing 'Barcode' column."

umap_df = umap_df.set_index("Barcode")

# align to adata.obs_names order
umap_df = umap_df.reindex(adata.obs_names)

# Optional check: make sure no missing values after reindex
if umap_df.isna().any().any():
    n_missing = umap_df.isna().any(axis=1).sum()
    print(f"Warning: {n_missing} cells have missing UMAP coordinates after reindex.")

# store in obsm with a clear R-specific name
umap_coords = umap_df[["UMAP_1", "UMAP_2"]].to_numpy(dtype="float32")
adata.obsm["X_umap_R"] = umap_coords

# Optionally, also put as obs columns if you like
adata.obs["UMAP_R_1"] = umap_coords[:, 0]
adata.obs["UMAP_R_2"] = umap_coords[:, 1]

print("obsm keys:", adata.obsm_keys())

# Summary:
# - Loaded the R UMAP embedding and aligned it to adata.obs_names.
# - Stored it in `adata.obsm['X_umap_R']` and in obs as `UMAP_R_1`, `UMAP_R_2`.


UMAP table shape: (135379, 3)
                    Barcode    UMAP_1    UMAP_2
0  56948_AAACGAACACTAGGTT-1  5.049781 -9.935757
1  56948_AAACGCTGTTGGGTTT-1  6.671258 -9.771027
2  56948_AAACGCTTCCTCGATC-1  7.841278  3.778570
3  56948_AAAGAACCAATACAGA-1  8.648595  2.988089
4  56948_AAAGAACCACAGGATG-1 -4.163362 -6.711728
obsm keys: ['X_umap_R']


In [6]:
# Cell 6: sanity checks + save

print(adata)
print("Number of cells per cell_type:")
if "cell_type" in adata.obs.columns:
    print(adata.obs["cell_type"].value_counts().head())
else:
    print("No 'cell_type' column found in obs.")

# Save h5ad for cell2location
out_file = os.path.join(base_dir, "sc_reference_cell2location.h5ad")
adata.write_h5ad(out_file, compression="gzip")

print("Saved AnnData to:", out_file)

# Summary:
# - Confirmed basic structure and cell_type distribution.
# - Wrote AnnData to `sc_reference_cell2location.h5ad` with R UMAP preserved.


AnnData object with n_obs × n_vars = 135379 × 22949
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Barcode', 'Library', 'condition', 'file_name', 'percent.mt', 'unintegrated_clusters', 'seurat_clusters', 'harmony_clusters', 'sctype_classification', 'seurat.cluster.ann', 'treatment', 'cell_type', 'UMAP_R_1', 'UMAP_R_2'
    obsm: 'X_umap_R'
Number of cells per cell_type:
cell_type
Monocyte      61613
Macrophage    31399
Fibroblast    18788
DC             6874
B_cell         5778
Name: count, dtype: int64
Saved AnnData to: /Users/janzules/Roselab/Spatial/CAR_T/data/sc-reference/sc_reference_cell2location.h5ad


# Sanity Check

In [1]:
import anndata as ad
file_loc = "/Users/janzules/Roselab/Spatial/CAR_T/data/sc-reference/sc_reference_cell2location.h5ad"
adata = ad.read_h5ad(file_loc)

## Checking defined top number of cells/genes

In [13]:
num = 15

In [14]:
cells = adata.obs_names[:num]
genes = adata.var_names[:num]

df = pd.DataFrame(
    adata.X[:15, :15].toarray(),
    index=cells,
    columns=genes
)

df


,Xkr4,Mrpl15,Lypla1,Gm37988,Tcea1,Rgs20,Atp6v1h,Rb1cc1,4732440D04Rik,St18,Pcmtd1,Gm26901,Sntg1,Rrs1,Adhfe1
56948_AAACGAACACTAGGTT-1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0
56948_AAACGCTGTTGGGTTT-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
56948_AAACGCTTCCTCGATC-1,0,0,0,0,0,0,3,1,1,11,22,0,0,0,0
56948_AAAGAACCAATACAGA-1,0,0,0,0,2,0,1,0,0,0,1,0,0,0,0
56948_AAAGAACCACAGGATG-1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
56948_AAAGAACTCCAAGAGG-1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0
56948_AAAGGATCACCAGCTG-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
56948_AAAGGTAGTCCGATCG-1,0,1,0,0,1,0,1,3,0,0,1,0,0,0,0
56948_AAAGTCCTCGTGCGAC-1,0,2,0,0,2,0,1,1,0,0,0,0,0,0,0
56948_AAAGTGACAGAGTTGG-1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


## Check the last

In [18]:
df_last = pd.DataFrame(
    adata.X[-15:, -15:].toarray(),
    index=adata.obs_names[-15:],
    columns=adata.var_names[-15:]
)
df_last

,Awat1,Gm13177,Gm15627,Serpina3b,Gm13467,4930466I24Rik,C2cd4c,Gm2670,Gm15640,Tceal7,Ntrk1,Gm28074,Gm12927,Sstr3,Smim22
56977_TTTGGAGGTTCATCGA-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
56977_TTTGGAGTCACGTCCT-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
56977_TTTGGAGTCTACCAGA-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
56977_TTTGGTTGTCGCTTAA-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
56977_TTTGGTTTCCACAGGC-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
56977_TTTGGTTTCTAGTCAG-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
56977_TTTGTTGAGTCGAAAT-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
56977_TTTGTTGAGTGCCAGA-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
56977_TTTGTTGCAAACTCGT-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
56977_TTTGTTGCAATGGGTG-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


# Checking random samples

In [12]:
import random
import numpy as np
import pandas as pd

In [5]:
# cells = adata.obs_names.tolist()
# selected_cells = random.sample(cells, 5)

# random.randint(1, len(cells))

In [6]:
X = adata.X
n_top = 10 # top number of cells
selected_cells = [36537, 107534, 104000]

In [5]:
# i = selected_cells[0]

In [7]:
cell_name = adata.obs_names[36537]

row = X[36537].toarray().ravel()

In [26]:
row

array([0, 0, 0, ..., 0, 0, 0])

In [30]:
test = X[36537].toarray().ravel() if hasattr(X[i], "toarray") else np.asarray(X[i]).ravel()

In [ ]:
np.argsort(test)[-n_top:][::-1]

In [16]:
np.argsort(row)[-n_top:][::-1]

array([14836, 16101, 16099, 16098, 16102, 14236,  6701,  2781, 10782,
        8387])

In [9]:
# np.argsort(row)[-n_top:][::-1]

In [9]:
tpg = np.argsort(row)[-n_top:][::-1]
tpg

array([ 3180, 16101, 16102, 16099, 14836, 16098,  7038, 15753, 13562,
        6001])

In [10]:
gens = adata.var_names[tpg]
vals_gens = row[tpg]

In [13]:
pd.DataFrame({"gene": gens, "counts":vals_gens})

,gene,counts
0,S100a6,70
1,mt-Atp6,51
2,mt-Co3,44
3,mt-Co2,35
4,Gm42418,29
5,mt-Co1,28
6,Ftl1,24
7,Fth1,17
8,Lgals1,15
9,Tmsb10,15


In [14]:
import numpy as np
import pandas as pd

X = adata.X  # usually sparse
n_top = 10   # number of top genes per cell

selected_cells = [36537, 107534, 104000]  # your cell indices

for i in selected_cells:
    cell_name = adata.obs_names[i]
    
    # row as dense 1D array
    row = X[i].toarray().ravel() if hasattr(X[i], "toarray") else np.asarray(X[i]).ravel()
    
    # indices of top genes (descending)
    top_idx = np.argsort(row)[-n_top:][::-1] # 
    print(i)
    print(top_idx)
    top_genes = adata.var_names[top_idx]
    top_vals  = row[top_idx]
    
    print(f"\n=== Cell index {i} | {cell_name} ===")
    print("Annotations:")
    print(adata.obs.loc[cell_name])  # or restrict columns
    
    print("\nTop genes:")
    df = pd.DataFrame({"gene": top_genes, "count": top_vals})
    print(df)


36537
[ 3180 16101 16102 16099 14836 16098  7038 15753 13562  6001]

=== Cell index 36537 | 56955_TGCAGATGTGAGCAGT-1 ===
Annotations:
orig.ident                        RTCyTAG72
nCount_RNA                             2390
nFeature_RNA                           1245
Barcode                  TGCAGATGTGAGCAGT-1
Library                               56955
condition                         RTCyTAG72
file_name                   RTCyTAG72_Tu2_1
percent.mt                         8.493724
unintegrated_clusters                     3
seurat_clusters                           2
harmony_clusters                          2
sctype_classification            Fibroblast
seurat.cluster.ann            2: Fibroblast
treatment                     RTCyTAG72_Tu2
cell_type                        Fibroblast
UMAP_R_1                          -5.642831
UMAP_R_2                          -7.824723
Name: 56955_TGCAGATGTGAGCAGT-1, dtype: object

Top genes:
      gene  count
0   S100a6     70
1  mt-Atp6     51
2   mt

## By barcode